In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q newspaper3k lxml_html_clean beautifulsoup4 requests tqdm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 57.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 95.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.4 MB/s eta 0:00:00


In [15]:
import pandas as pd

file_path = "/kaggle/input/datasets/chshani2/new-backlink-files/Vaseljenska.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (100, 7)
Columns: ['Result', 'URL', 'Anchor text', 'Nofollow', 'Unnamed: 4', 'Total links', 'Landing page']


,Result,URL,Anchor text,Nofollow,Unnamed: 4,Total links,Landing page
0,1,https://myslpolska.info/2026/01/23/wegry-ukrai...,dokumentów,False,59,NaN,https://vaseljenska.net/2026/01/05/ekskluzivna...
1,2,https://en.wikipedia.org/wiki/List_of_active_s...,"""""""СКАНДАЛ СВИ СЕПАРАТИСТИ НА ЈЕДНОМ МЕСТУ: На...",True,53,NaN,https://vaseljenska.net/2023/03/03/skandal-svi...
2,3,https://novivesnik.net/2023/08/28/predlog-rezo...,Вучић: Опозиција ће имати изборе и пре него шт...,True,47,NaN,https://vaseljenska.net/2023/08/29/vucic-opozi...
3,4,https://magyarnemzet.hu/kulfold/2025/09/zelens...,Vaseljenska szerb portál birtokába került bels...,False,46,NaN,https://vaseljenska.net/2025/09/11/ekskluzivno...
4,5,https://mandiner.hu/kulfold/2026/01/kiszivargo...,Vaseljenska nevű oldal,False,35,NaN,https://vaseljenska.net/2026/01/05/ekskluzivna...


In [18]:
# Clean column names only for easier handling
df = df.rename(columns={
    "URL": "source_url",
    "Landing page": "target_url"
})

# Keep only valid URLs
source_urls = df["source_url"].dropna().astype(str).str.strip()
target_urls = df["target_url"].dropna().astype(str).str.strip()

source_urls = source_urls[source_urls.str.startswith("http")]
target_urls = target_urls[target_urls.str.startswith("http")]

# Unique source and target URLs separately
unique_source_urls = source_urls.drop_duplicates().tolist()
unique_target_urls = target_urls.drop_duplicates().tolist()

print("Unique source URLs:", len(unique_source_urls))
print("Unique target URLs:", len(unique_target_urls))

Unique source URLs: 100
Unique target URLs: 78


# ***Extract text for SOURCE URLs***

In [19]:
import requests
from bs4 import BeautifulSoup
from newspaper import Article
from tqdm import tqdm
import time

def extract_text_from_url(url, timeout=15):
    """
    Extract text from a URL using newspaper3k first.
    If it fails, use BeautifulSoup as fallback.
    """

    # Method 1: newspaper3k
    try:
        article = Article(url)
        article.download()
        article.parse()

        text = article.text.strip()

        if len(text) > 200:
            return text, "newspaper3k", "success"

    except Exception as e:
        newspaper_error = type(e).__name__

    # Method 2: BeautifulSoup fallback
    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            )
        }

        response = requests.get(url, headers=headers, timeout=timeout)

        if response.status_code != 200:
            return "", "beautifulsoup", f"status_{response.status_code}"

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove noisy page elements
        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            tag.decompose()

        text = " ".join(soup.get_text(separator=" ").split())

        if len(text) > 200:
            return text, "beautifulsoup", "success"

        return "", "beautifulsoup", "too_short"

    except Exception as e:
        return "", "failed", type(e).__name__

In [20]:
def extract_url_list(url_list, url_type, max_urls=None, sleep_time=0.3):
    """
    Extract text for a list of URLs.
    url_type should be 'source' or 'target'.
    """

    if max_urls is not None:
        url_list = url_list[:max_urls]

    results = []

    for url in tqdm(url_list, desc=f"Extracting {url_type} URLs"):
        text, method, status = extract_text_from_url(url)

        results.append({
            f"{url_type}_url": url,
            f"{url_type}_text": text,
            f"{url_type}_text_length": len(text),
            f"{url_type}_extraction_method": method,
            f"{url_type}_extraction_status": status
        })

        time.sleep(sleep_time)

    return pd.DataFrame(results)

In [21]:
MAX_SOURCE_URLS = None  # change to None for all source URLs

source_text_df = extract_url_list(
    unique_source_urls,
    url_type="source",
    max_urls=MAX_SOURCE_URLS
)

source_text_df.head()

Extracting source URLs: 100%|██████████| 100/100 [03:27<00:00,  2.07s/it]


,source_url,source_text,source_text_length,source_extraction_method,source_extraction_status
0,https://myslpolska.info/2026/01/23/wegry-ukrai...,"Relacje między Węgrami a Ukrainą, dalekie od n...",2763,newspaper3k,success
1,https://en.wikipedia.org/wiki/List_of_active_s...,Supporters of Catalan independence in Barcelon...,1979,newspaper3k,success
2,https://novivesnik.net/2023/08/28/predlog-rezo...,PREDLOG REZOLUCIJE O PRIDRUŽIVANJU BRIКS-u\n\n...,7683,newspaper3k,success
3,https://magyarnemzet.hu/kulfold/2025/09/zelens...,Zelenszkij Magyarország ellen indítana kampány...,2272,newspaper3k,success
4,https://mandiner.hu/kulfold/2026/01/kiszivargo...,A szerb Vaseljenska nevű oldal nyilvánosságra ...,326,newspaper3k,success


In [9]:
source_text_df.to_csv("/kaggle/working/euronews_source_url_text.csv", index=False)

print("Source URL text saved.")

Source URL text saved.


# ***Extract text for TARGET URLs***

In [22]:
MAX_TARGET_URLS = None # change to None for all target URLs

target_text_df = extract_url_list(
    unique_target_urls,
    url_type="target",
    max_urls=MAX_TARGET_URLS
)

target_text_df.head()

Extracting target URLs: 100%|██████████| 78/78 [17:01<00:00, 13.10s/it]


,target_url,target_text,target_text_length,target_extraction_method,target_extraction_status
0,https://vaseljenska.net/2026/01/05/ekskluzivna...,ЕКСКЛУЗИВНА ДОКУМЕНТА СБУ: Зеленски наводно на...,7208,beautifulsoup,success
1,https://vaseljenska.net/2023/03/03/skandal-svi...,СКАНДАЛ СВИ СЕПАРАТИСТИ НА ЈЕДНОМ МЕСТУ: На пр...,8653,beautifulsoup,success
2,https://vaseljenska.net/2023/08/29/vucic-opozi...,Вучић: Опозиција ће имати изборе и пре него шт...,5090,beautifulsoup,success
3,https://vaseljenska.net/2025/09/11/ekskluzivno...,ЕКСКЛУЗИВНО ОТКРИВАМО: Документ који потреса Е...,20445,beautifulsoup,success
4,https://vaseljenska.net/,Почетна - Васељенска ТВ Skip to content НАЈНОВ...,7800,beautifulsoup,success


In [12]:
target_text_df.to_csv("/kaggle/working/RT_news_target_url_text.csv", index=False)

print("Target URL text saved.")

Target URL text saved.


In [23]:
# =========================
# Merge only processed source/target URL rows
# =========================

# URLs that were actually processed
processed_source_urls = set(source_text_df["source_url"].dropna().unique())
processed_target_urls = set(target_text_df["target_url"].dropna().unique())

# Keep only rows from original df where source OR target URL was processed
df_limited = df[
    df["source_url"].isin(processed_source_urls) |
    df["target_url"].isin(processed_target_urls)
].copy()

print("Original df shape:", df.shape)
print("Limited df shape:", df_limited.shape)

# Merge source text
merged_df = df_limited.merge(
    source_text_df,
    on="source_url",
    how="left"
)

# Merge target text
merged_df = merged_df.merge(
    target_text_df,
    on="target_url",
    how="left"
)

print("Merged shape:", merged_df.shape)

merged_df.head()

Original df shape: (100, 7)
Limited df shape: (100, 7)
Merged shape: (100, 15)


,Result,source_url,Anchor text,Nofollow,Unnamed: 4,Total links,target_url,source_text,source_text_length,source_extraction_method,source_extraction_status,target_text,target_text_length,target_extraction_method,target_extraction_status
0,1,https://myslpolska.info/2026/01/23/wegry-ukrai...,dokumentów,False,59,NaN,https://vaseljenska.net/2026/01/05/ekskluzivna...,"Relacje między Węgrami a Ukrainą, dalekie od n...",2763,newspaper3k,success,ЕКСКЛУЗИВНА ДОКУМЕНТА СБУ: Зеленски наводно на...,7208,beautifulsoup,success
1,2,https://en.wikipedia.org/wiki/List_of_active_s...,"""""""СКАНДАЛ СВИ СЕПАРАТИСТИ НА ЈЕДНОМ МЕСТУ: На...",True,53,NaN,https://vaseljenska.net/2023/03/03/skandal-svi...,Supporters of Catalan independence in Barcelon...,1979,newspaper3k,success,СКАНДАЛ СВИ СЕПАРАТИСТИ НА ЈЕДНОМ МЕСТУ: На пр...,8653,beautifulsoup,success
2,3,https://novivesnik.net/2023/08/28/predlog-rezo...,Вучић: Опозиција ће имати изборе и пре него шт...,True,47,NaN,https://vaseljenska.net/2023/08/29/vucic-opozi...,PREDLOG REZOLUCIJE O PRIDRUŽIVANJU BRIКS-u\n\n...,7683,newspaper3k,success,Вучић: Опозиција ће имати изборе и пре него шт...,5090,beautifulsoup,success
3,4,https://magyarnemzet.hu/kulfold/2025/09/zelens...,Vaseljenska szerb portál birtokába került bels...,False,46,NaN,https://vaseljenska.net/2025/09/11/ekskluzivno...,Zelenszkij Magyarország ellen indítana kampány...,2272,newspaper3k,success,ЕКСКЛУЗИВНО ОТКРИВАМО: Документ који потреса Е...,20445,beautifulsoup,success
4,5,https://mandiner.hu/kulfold/2026/01/kiszivargo...,Vaseljenska nevű oldal,False,35,NaN,https://vaseljenska.net/2026/01/05/ekskluzivna...,A szerb Vaseljenska nevű oldal nyilvánosságra ...,326,newspaper3k,success,ЕКСКЛУЗИВНА ДОКУМЕНТА СБУ: Зеленски наводно на...,7208,beautifulsoup,success


In [24]:
merged_output_path = "/kaggle/working/Vaseljenska_source_and_target_text_1.csv"

merged_df.to_csv(merged_output_path, index=False)

print("Saved limited merged file at:", merged_output_path)

Saved limited merged file at: /kaggle/working/Vaseljenska_source_and_target_text_1.csv


In [35]:
source_success = source_text_df[source_text_df["source_text_length"] > 200].shape[0]
target_success = target_text_df[target_text_df["target_text_length"] > 200].shape[0]

print("Source URLs processed:", len(source_text_df))
print("Source URLs successfully extracted:", source_success)

print("Target URLs processed:", len(target_text_df))
print("Target URLs successfully extracted:", target_success)

Source URLs processed: 500
Source URLs successfully extracted: 467
Target URLs processed: 500
Target URLs successfully extracted: 499


# ***Output Level Prediction***

In [21]:
!pip install -q gradio openpyxl

In [22]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

from transformers import BertTokenizer, BertForSequenceClassification # Changed to import specific classes
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from urllib.parse import urlparse

import gradio as gr

# 1) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2) Load fine-tuned model + tokenizer
MODEL_PATH = "Zeeshanshanih/mbert-misinformation-detector"  # <-- change this path

# Explicitly load BertTokenizer and BertForSequenceClassification
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

print("Loaded model from:", MODEL_PATH)
print("Num labels:", model.config.num_labels)


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded model from: Zeeshanshanih/mbert-misinformation-detector
Num labels: 3


In [23]:
MAX_LENGTH = 256  # same as training

def predict_texts(text_list, batch_size=16):
    all_probs = []
    all_preds = []

    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i+batch_size]

        # ensure strings
        batch_texts = ["" if x is None else str(x) for x in batch_texts]

        encodings = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits                      # [B, num_labels]
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

        all_probs.extend(probs)
        all_preds.extend(preds)

    return np.array(all_probs), np.array(all_preds)


In [24]:
def analyze_backlinks_df(
    df: pd.DataFrame,
    content_thresh: float = 0.6,
    network_thresh: float = 0.6,
    source_col: str = "source_text",
    target_col: str = "target_text",
    url_col: str    = "target_url"
):
    # --- 1) Basic checks ---
    for col in [source_col, target_col, url_col]:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in uploaded file.")

    # Ensure a row_index
    if "row_index" not in df.columns:
        df = df.copy()
        df["row_index"] = np.arange(len(df))
    else:
        df = df.copy()

    # --- 2) Run predictions for source & target texts ---
    source_texts = df[source_col].fillna("").tolist()
    target_texts = df[target_col].fillna("").tolist()

    source_probs, source_preds = predict_texts(source_texts, batch_size=16)
    df["source_pred"]      = source_preds
    df["source_prob_prop"] = source_probs[:, 1]

    target_probs, target_preds = predict_texts(target_texts, batch_size=16)
    df["target_pred"]      = target_preds
    df["target_prob_prop"] = target_probs[:, 1]

    # --- 3) Extract outlet_domain from target_url ---
    def extract_domain(url):
        try:
            netloc = urlparse(str(url)).netloc
            if netloc.startswith("www."):
                netloc = netloc[4:]
            return netloc
        except Exception:
            return ""

    df["outlet_domain"] = df[url_col].apply(extract_domain)

    # --- 4) PAGE-LEVEL aggregation (per target_url) ---
    def first_label(series):
        return series.iloc[0] if len(series) > 0 else np.nan

    page_stats = (
        df
        .groupby("target_url")
        .agg(
            outlet_domain=("outlet_domain", "first"),
            n_links_page=("row_index", "count"),
            page_content_label=("target_pred", first_label),
            page_content_score=("target_prob_prop", "mean"),
            page_incoming_prop_ratio=("source_pred",
                                      lambda s: (s == 1).mean() if len(s) > 0 else np.nan),
            page_avg_source_prob=("source_prob_prop", "mean"),
        )
        .reset_index()
    )

    # --- 5) OUTLET-LEVEL aggregation (over pages) ---
    outlet_stats = (
        page_stats
        .groupby("outlet_domain")
        .agg(
            n_pages=("target_url", "count"),
            n_links=("n_links_page", "sum"),
            target_prop_ratio=("page_content_label",
                               lambda s: (s == 1).mean() if len(s) > 0 else np.nan),
            avg_target_prob_prop=("page_content_score", "mean"),
            incoming_prop_ratio=("page_incoming_prop_ratio", "mean"),
            avg_source_prob_prop=("page_avg_source_prob", "mean"),
        )
        .reset_index()
    )

    # --- 6) Classify outlets ---
    def classify_outlet(row):
        content_high = row["target_prop_ratio"]   >= content_thresh
        network_high = row["incoming_prop_ratio"] >= network_thresh

        if content_high and network_high:
            return "propaganda_outlet"
        elif content_high and not network_high:
            return "content_leaning_propaganda"
        elif network_high and not content_high:
            return "network_leaning_propaganda"
        else:
            return "mainly_legitimate"

    outlet_stats["outlet_label"] = outlet_stats.apply(classify_outlet, axis=1)

    # --- 7) Round for display ---
    for col in ["target_prop_ratio", "incoming_prop_ratio",
                "avg_source_prob_prop", "avg_target_prob_prop"]:
        if col in outlet_stats.columns:
            outlet_stats[col] = outlet_stats[col].round(3)

    return df, outlet_stats


In [25]:
def analyze_backlinks_app(file_obj, content_thresh, network_thresh):
    if file_obj is None:
        return pd.DataFrame(), pd.DataFrame()

    filename = file_obj.name
    if filename.endswith(".csv"):
        df = pd.read_csv(file_obj)
    elif filename.endswith(".xlsx") or filename.endswith(".xls"):
        df = pd.read_excel(file_obj)
    else:
        raise ValueError("Please upload a .csv or .xlsx file.")

    # Run full analysis (row-level + outlet-level)
    link_df, outlet_df = analyze_backlinks_df(
        df,
        content_thresh=content_thresh,
        network_thresh=network_thresh,
        source_col="source_text",
        target_col="target_text",
        url_col="target_url"
    )

    # Row-level display (narrower set of columns)
    display_cols_link = [
        "row_index", "source_url", "source_pred", "source_prob_prop",
        "target_url", "target_pred", "target_prob_prop", "outlet_domain"
    ]
    display_cols_link = [c for c in display_cols_link if c in link_df.columns]
    link_df_display = link_df[display_cols_link]

    # Outlet-level display
    display_cols_outlet = [
        "outlet_domain", "n_links", "target_prop_ratio",
        "incoming_prop_ratio", "avg_source_prob_prop",
        "avg_target_prob_prop", "outlet_label"
    ]
    display_cols_outlet = [c for c in display_cols_outlet if c in outlet_df.columns]
    outlet_df_display = outlet_df[display_cols_outlet]

    return link_df_display, outlet_df_display


In [26]:
with gr.Blocks() as demo:
    gr.Markdown("## Hybrid Backlink–NLP Outlet Credibility Analyzer")

    # --- Input section (top) ---
    with gr.Row():
        file_input = gr.File(
            label="Backlinks file (.csv or .xlsx)",
            file_types=[".csv", ".xlsx", ".xls"]
        )

    with gr.Row():
        with gr.Column():
            content_slider = gr.Slider(
                0.0, 1.0, value=0.6, step=0.05,
                label="Content threshold"
            )
        with gr.Column():
            network_slider = gr.Slider(
                0.0, 1.0, value=0.6, step=0.05,
                label="Network threshold"
            )
        run_btn = gr.Button("Run analysis", variant="primary")

    gr.Markdown("### Row-level predictions (source + target pages)")
    row_table = gr.Dataframe(
        interactive=False,
        label="Row-level predictions"
    )

    gr.Markdown("### Outlet-level summary (credibility labels)")
    outlet_table = gr.Dataframe(
        interactive=False,
        label="Outlet-level summary"
    )

    # Wire button to function
    run_btn.click(
        fn=analyze_backlinks_app,
        inputs=[file_input, content_slider, network_slider],
        outputs=[row_table, outlet_table]
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://85129f4dfae2eed92c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

# ***Outlet Level Indicators Updated***

In [80]:
!pip install -q transformers torch tqdm tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.1 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [81]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import tldextract

In [83]:
file_path = "/kaggle/working/BBC_news_source_and_target_text_1.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (100, 15)
Columns: ['Result', 'source_url', 'Anchor text', 'Nofollow', 'Unnamed: 4', 'Total links', 'target_url', 'source_text', 'source_text_length', 'source_extraction_method', 'source_extraction_status', 'target_text', 'target_text_length', 'target_extraction_method', 'target_extraction_status']


,Result,source_url,Anchor text,Nofollow,Unnamed: 4,Total links,target_url,source_text,source_text_length,source_extraction_method,source_extraction_status,target_text,target_text_length,target_extraction_method,target_extraction_status
0,1,https://www.lestinto.ch/,la vendita di,False,93,NaN,https://www.bbc.com/news/uk-65932372,"Nel 2019 Luigi Di Maio, che all’epoca era vice...",2434,newspaper3k,success,"""The speed with which these emerging technolog...",301,newspaper3k,success
1,2,https://www.bbc.co.uk/aboutthebbc,BBC Account,False,92,NaN,https://account.bbc.com/account,The BBC is the world’s leading public service ...,1696,newspaper3k,success,NaN,0,beautifulsoup,too_short
2,3,https://advertising.bbcstudios.com/,BBC.com,False,91,NaN,https://www.bbc.com/,Who We Are\n\nAs the global storytelling power...,565,newspaper3k,success,The people who are addicted to daydreaming\n\n...,220,newspaper3k,success
3,4,https://www.timesofisrael.com/,Iran Strait of Hormuz warning adds to shipping...,False,90,NaN,https://www.bbc.com/news/articles/c3w39lg84w2o...,NaN,0,beautifulsoup,status_403,"""You've had nearly 800 ships stuck in there fo...",202,newspaper3k,success
4,5,https://www.newsflix.at/,mit Farbe und Graffitis,True,90,NaN,https://www.bbc.com/news/articles/c07jm1kp228o,Worum geht es? Die USA gaben am Montag bekannt...,1802,newspaper3k,success,Palestine Action arrest after Winston Churchil...,4854,beautifulsoup,success


In [85]:
required_cols = ["source_url", "target_url", "source_text", "target_text"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

print("All required columns are present.")

All required columns are present.


In [86]:
MODEL_NAME = "Zeeshanshanih/mbert-misinformation-detector"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

print("Model loaded successfully.")

Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.


In [88]:
def predict_batch(texts, batch_size=16, max_length=512):
    """
    Predict labels and propaganda probabilities for a list of texts.
    Returns:
    - predicted labels: 0 legitimate, 1 propaganda
    - propaganda probabilities
    """

    all_preds = []
    all_prop_probs = []

    texts = [str(t) if pd.notna(t) else "" for t in texts]

    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)

        prop_probs = probs[:, 1].detach().cpu().numpy()
        preds = np.argmax(probs.detach().cpu().numpy(), axis=1)

        all_preds.extend(preds.tolist())
        all_prop_probs.extend(prop_probs.tolist())

    return all_preds, all_prop_probs

In [94]:
MIN_TEXT_LENGTH = 200

df["source_text"] = df["source_text"].fillna("").astype(str)
df["target_text"] = df["target_text"].fillna("").astype(str)

df["source_text_length"] = df["source_text"].str.len()
df["target_text_length"] = df["target_text"].str.len()

df_valid = df[
    (df["source_text_length"] >= MIN_TEXT_LENGTH) &
    (df["target_text_length"] >= MIN_TEXT_LENGTH)
].copy()

print("Original rows:", len(df))
print("Valid rows after text filtering:", len(df_valid))

df_valid.head()

Original rows: 100
Valid rows after text filtering: 83


,Result,source_url,Anchor text,Nofollow,Unnamed: 4,Total links,target_url,source_text,source_text_length,source_extraction_method,source_extraction_status,target_text,target_text_length,target_extraction_method,target_extraction_status
0,1,https://www.lestinto.ch/,la vendita di,False,93,NaN,https://www.bbc.com/news/uk-65932372,"Nel 2019 Luigi Di Maio, che all’epoca era vice...",2434,newspaper3k,success,"""The speed with which these emerging technolog...",301,newspaper3k,success
2,3,https://advertising.bbcstudios.com/,BBC.com,False,91,NaN,https://www.bbc.com/,Who We Are\n\nAs the global storytelling power...,565,newspaper3k,success,The people who are addicted to daydreaming\n\n...,220,newspaper3k,success
4,5,https://www.newsflix.at/,mit Farbe und Graffitis,True,90,NaN,https://www.bbc.com/news/articles/c07jm1kp228o,Worum geht es? Die USA gaben am Montag bekannt...,1802,newspaper3k,success,Palestine Action arrest after Winston Churchil...,4854,beautifulsoup,success
5,6,https://nbot.ai/,Trump's attack on former allies exposes splint...,False,89,NaN,https://www.bbc.com/news/articles/cz0ep709pr2o,Everything you need to know about your new AI ...,1829,newspaper3k,success,"Prior to parting ways with Fox News in 2023, C...",403,newspaper3k,success
7,8,https://www.epochtimes.com/,BBC,False,88,NaN,https://www.bbc.com/zhongwen/simp,大纪元 | 大纪元新闻网 头条 1/12 美伊谈判为何走走停停 卢比奥有解释 头条 2/12...,7836,beautifulsoup,success,"主页 - BBC News 中文 BBC News , 中文 - 主页 头条新闻 分析：特朗...",3704,beautifulsoup,success


In [95]:
source_preds, source_prop_probs = predict_batch(
    df_valid["source_text"].tolist(),
    batch_size=16
)

df_valid["source_pred"] = source_preds
df_valid["source_prob_prop"] = source_prop_probs

df_valid[["source_url", "source_pred", "source_prob_prop"]].head()

Predicting: 100%|██████████| 6/6 [00:02<00:00,  2.30it/s]


,source_url,source_pred,source_prob_prop
0,https://www.lestinto.ch/,1,0.999995
2,https://advertising.bbcstudios.com/,1,0.999630
4,https://www.newsflix.at/,1,0.816754
5,https://nbot.ai/,0,0.350071
7,https://www.epochtimes.com/,1,0.999999


In [96]:
target_preds, target_prop_probs = predict_batch(
    df_valid["target_text"].tolist(),
    batch_size=16
)

df_valid["target_pred"] = target_preds
df_valid["target_prob_prop"] = target_prop_probs

df_valid[["target_url", "target_pred", "target_prob_prop"]].head()

Predicting: 100%|██████████| 6/6 [00:02<00:00,  2.42it/s]


,target_url,target_pred,target_prob_prop
0,https://www.bbc.com/news/uk-65932372,1,0.998645
2,https://www.bbc.com/,1,0.999999
4,https://www.bbc.com/news/articles/c07jm1kp228o,1,0.999999
5,https://www.bbc.com/news/articles/cz0ep709pr2o,0,0.000001
7,https://www.bbc.com/zhongwen/simp,1,0.999999


In [97]:
label_map = {
    0: "legitimate",
    1: "propaganda"
}

df_valid["source_label"] = df_valid["source_pred"].map(label_map)
df_valid["target_label"] = df_valid["target_pred"].map(label_map)

df_valid.head()

,Result,source_url,Anchor text,Nofollow,Unnamed: 4,Total links,target_url,source_text,source_text_length,source_extraction_method,...,target_text,target_text_length,target_extraction_method,target_extraction_status,source_pred,source_prob_prop,target_pred,target_prob_prop,source_label,target_label
0,1,https://www.lestinto.ch/,la vendita di,False,93,NaN,https://www.bbc.com/news/uk-65932372,"Nel 2019 Luigi Di Maio, che all’epoca era vice...",2434,newspaper3k,...,"""The speed with which these emerging technolog...",301,newspaper3k,success,1,0.999995,1,0.998645,propaganda,propaganda
2,3,https://advertising.bbcstudios.com/,BBC.com,False,91,NaN,https://www.bbc.com/,Who We Are\n\nAs the global storytelling power...,565,newspaper3k,...,The people who are addicted to daydreaming\n\n...,220,newspaper3k,success,1,0.999630,1,0.999999,propaganda,propaganda
4,5,https://www.newsflix.at/,mit Farbe und Graffitis,True,90,NaN,https://www.bbc.com/news/articles/c07jm1kp228o,Worum geht es? Die USA gaben am Montag bekannt...,1802,newspaper3k,...,Palestine Action arrest after Winston Churchil...,4854,beautifulsoup,success,1,0.816754,1,0.999999,propaganda,propaganda
5,6,https://nbot.ai/,Trump's attack on former allies exposes splint...,False,89,NaN,https://www.bbc.com/news/articles/cz0ep709pr2o,Everything you need to know about your new AI ...,1829,newspaper3k,...,"Prior to parting ways with Fox News in 2023, C...",403,newspaper3k,success,0,0.350071,0,0.000001,legitimate,legitimate
7,8,https://www.epochtimes.com/,BBC,False,88,NaN,https://www.bbc.com/zhongwen/simp,大纪元 | 大纪元新闻网 头条 1/12 美伊谈判为何走走停停 卢比奥有解释 头条 2/12...,7836,beautifulsoup,...,"主页 - BBC News 中文 BBC News , 中文 - 主页 头条新闻 分析：特朗...",3704,beautifulsoup,success,1,0.999999,1,0.999999,propaganda,propaganda


In [98]:
row_output_path = "/kaggle/working/backlink_row_level_predictions.csv"

df_valid.to_csv(row_output_path, index=False)

print("Row-level predictions saved at:", row_output_path)

Row-level predictions saved at: /kaggle/working/backlink_row_level_predictions.csv


In [101]:
page_level = (
    df_valid
    .groupby("target_url")
    .agg(
        page_content_label=("target_pred", "first"),
        page_content_score=("target_prob_prop", "first"),
        n_incoming_links=("source_url", "count"),
        n_prop_sources=("source_pred", lambda x: (x == 1).sum()),
        page_incoming_prop_ratio=("source_pred", lambda x: (x == 1).mean()),
        page_avg_source_prob=("source_prob_prop", "mean")
    )
    .reset_index()
)

page_level.head()

,target_url,page_content_label,page_content_score,n_incoming_links,n_prop_sources,page_incoming_prop_ratio,page_avg_source_prob
0,http://www.bbc.com/,1,0.999999,1,1,1.0,0.999999
1,http://www.bbc.com/mundo/,1,0.999986,1,1,1.0,0.999999
2,http://www.bbc.com/news,1,0.999999,1,1,1.0,0.999995
3,http://www.bbc.com/news/technology-28106815,1,0.999998,1,1,1.0,0.999999
4,http://www.bbc.com/news/technology-33989384,0,0.078231,1,0,0.0,0.000006


In [106]:
def extract_domain(url):
    try:
        ext = tldextract.extract(str(url))
        if ext.domain and ext.suffix:
            return f"{ext.domain}.{ext.suffix}"
        return ""
    except:
        return ""

page_level["outlet_domain"] = page_level["target_url"].apply(extract_domain)

page_level[["target_url", "outlet_domain"]].head()

,target_url,outlet_domain
0,http://www.bbc.com/,bbc.com
1,http://www.bbc.com/mundo/,bbc.com
2,http://www.bbc.com/news,bbc.com
3,http://www.bbc.com/news/technology-28106815,bbc.com
4,http://www.bbc.com/news/technology-33989384,bbc.com


In [108]:
outlet_level = (
    page_level
    .groupby("outlet_domain")
    .agg(
        n_pages=("target_url", "count"),
        n_links=("n_incoming_links", "sum"),

        target_prop_ratio=("page_content_label", lambda x: (x == 1).mean()),
        avg_target_prob_prop=("page_content_score", "mean"),

        incoming_prop_ratio=("page_incoming_prop_ratio", "mean"),
        avg_source_prob_prop=("page_avg_source_prob", "mean")
    )
    .reset_index()
)

outlet_level.head()

,outlet_domain,n_pages,n_links,target_prop_ratio,avg_target_prob_prop,incoming_prop_ratio,avg_source_prob_prop
0,bbc.com,79,83,0.848101,0.854201,0.898734,0.900309


In [109]:
TAU_CONTENT = 0.6
TAU_NETWORK = 0.6

def assign_outlet_label(row):
    content_high = row["target_prop_ratio"] >= TAU_CONTENT
    network_high = row["incoming_prop_ratio"] >= TAU_NETWORK

    if content_high and network_high:
        return "content-and-network propaganda"
    elif content_high and not network_high:
        return "content-leaning propaganda"
    elif not content_high and network_high:
        return "network-leaning propaganda"
    else:
        return "mostly legitimate"

outlet_level["outlet_label"] = outlet_level.apply(assign_outlet_label, axis=1)

outlet_level

,outlet_domain,n_pages,n_links,target_prop_ratio,avg_target_prob_prop,incoming_prop_ratio,avg_source_prob_prop,outlet_label
0,bbc.com,79,83,0.848101,0.854201,0.898734,0.900309,content-and-network propaganda
